<a href="https://colab.research.google.com/github/fphsFischmeister/ILAE_NeuroimagingSchool/blob/master/notebooks/06_connectivity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Google Colab"/>  </a>

# Welcome to the interactive ILAE workshop on task-based activation detection.

Author: Florian Ph.S Fischmeister, Marc Berger, Radeshyam Stepponat

---
- Part 1: Preprocessing fMRI data with fMRIPrep [Jupyter Notebook](https://colab.research.google.com/github/fphsFischmeister/ILAE_NeuroimagingSchool/blob/master/notebooks/01_preprocessing.ipynb)
- Part 2: First Level of a simple motor task [Jupyter Notebook](https://colab.research.google.com/github/fphsFischmeister/ILAE_NeuroimagingSchool/blob/master/notebooks/02_basic_motor_task.ipynb)
- Part 3: A Home-Town-Walking language paradigm [Jupyter Notebook](https://colab.research.google.com/github/fphsFischmeister/ILAE_NeuroimagingSchool/blob/master/notebooks/03_hometown_task.ipynb)
- Part 4: Phrases, a language paradigm [Jupyter Notebook](https://colab.research.google.com/github/fphsFischmeister/ILAE_NeuroimagingSchool/blob/master/notebooks/04_phases_task.ipynb)
- Part 5: All language tasks [Jupyter Notebook](https://colab.research.google.com/github/fphsFischmeister/ILAE_NeuroimagingSchool/blob/master/notebooks/05_all_language_task.ipynb)
- Part 6: Functional Connectivity [Jupyter Notebook](https://colab.research.google.com/github/fphsFischmeister/ILAE_NeuroimagingSchool/blob/master/notebooks/06_connectivity.ipynb)



In [104]:
# get some data for presentation
!rm -rf ILAE_NeuroimagingSchool
!git clone https://github.com/fphsFischmeister/ILAE_NeuroimagingSchool.git

# get some functional motor data
!wget https://dinlab.roentgen.meduniwien.ac.at/ILAE_NeuroimagingSchool/dataset/sub-ILAEDemo001_ses-01_task-Rest_run-01_space-MNI_desc-denoised_bold.nii.gz -P ILAE_NeuroimagingSchool/dataset/func/


Cloning into 'ILAE_NeuroimagingSchool'...
remote: Enumerating objects: 259, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 259 (delta 0), reused 1 (delta 0), pack-reused 254 (from 1)
Receiving objects: 100% (259/259), 201.71 MiB | 25.01 MiB/s, done.
Resolving deltas: 100% (120/120), done.
--2026-05-05 21:37:04--  https://dinlab.roentgen.meduniwien.ac.at/ILAE_NeuroimagingSchool/dataset/sub-ILAEDemo001_ses-01_task-Rest_run-01_space-MNI_desc-denoised_bold.nii.gz
Resolving dinlab.roentgen.meduniwien.ac.at (dinlab.roentgen.meduniwien.ac.at)... 149.148.226.8
Connecting to dinlab.roentgen.meduniwien.ac.at (dinlab.roentgen.meduniwien.ac.at)|149.148.226.8|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 909146296 (867M) [application/octet-stream]
Saving to: ‘ILAE_NeuroimagingSchool/dataset/func/sub-ILAEDemo001_ses-01_task-Rest_run-01_space-MNI_desc-denoised_bold.nii.gz’

sub-ILAEDemo001_ses 100%[===========

In [105]:
# install modules
%pip install nilearn
%pip install ipyniivue
%pip install ipywidgets
%pip install matplotlib

import numpy as np
import pandas as pd
import nibabel as nib
import nilearn
from pathlib import Path

/home/fischmei/ILAE_NeuroimagingSchool/.venv/bin/python3: No module named pip
Note: you may need to restart the kernel to use updated packages.
/home/fischmei/ILAE_NeuroimagingSchool/.venv/bin/python3: No module named pip
Note: you may need to restart the kernel to use updated packages.
/home/fischmei/ILAE_NeuroimagingSchool/.venv/bin/python3: No module named pip
Note: you may need to restart the kernel to use updated packages.
/home/fischmei/ILAE_NeuroimagingSchool/.venv/bin/python3: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [106]:
# some basic analysis definitions for later use
task_label = "Rest"

nifti_filename = "./ILAE_NeuroimagingSchool/dataset/func/sub-ILAEDemo001_ses-01_task-Rest_run-01_space-MNI_desc-denoised_bold.nii.gz"
nifti_filename_new = "./ILAE_NeuroimagingSchool/outputs/sub-ILAEDemo001_ses-01_task-Rest_run-01_space-MNI_desc-noNAN_bold.nii.gz"

confounds_filename = "./ILAE_NeuroimagingSchool/dataset/func/sub-ILAEDemo001_ses-01_task-Rest_run-01_desc-confounds_timeseries.tsv"

!mkdir -p "./ILAE_NeuroimagingSchool/outputs/"
output_dir = "./ILAE_NeuroimagingSchool/outputs/"



In [107]:
# need to replace NaNs with 0 for nilearn to work properly, otherwise we get an error about "Input contains NaN, infinity or a value too large for dtype('float64')."
from nilearn import image

# 1. Load the Nifti image
img = nib.load(nifti_filename)

# 2. Get data and replace NaNs with 0
data = img.get_fdata()
data_no_nan = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)

# 3. Create a new Nifti image with the same affine
new_img = nib.Nifti1Image(data_no_nan, img.affine)

# 4. Save the new image
nib.save(new_img, nifti_filename_new)

# Part 6: Connectome of a simple Resting-State


In this notebook, we perform a -level task-based fMRI analysis of a simple language paradigm using Nilearn. This analysis builds on the preprocessing notebook: we assume that the raw MRI data have already been preprocessed with fMRIPrep and that the relevant outputs are available in BIDS-Derivatives format. The main inputs are the preprocessed BOLD image, the task events file, the brain mask, and selected confound regressors from fMRIPrep. 


The goal is to estimate, for one participant and one functional run, which brain regions show BOLD signal changes associated with the language task. To do this, we fit a first-level general linear model, or GLM, which is applied voxel by voxel. The model compares the measured BOLD time series with a predicted BOLD response derived from the experimental paradigm. Events such as phrases in this case are described in an *events.tsv* file, transformed into predicted BOLD responses by convolution with a haemodynamic response function, or HRF, and combined into a design matrix. The statistical analysis then tests whether a contrast of design-matrix columns explains a significant proportion of the fMRI signal at each brain location.

https://peerherholz.github.io/workshop_weizmann/advanced/functional_connectivity.html


In [ ]:
## XCP-D Worklow

(./ILAE)

## Brain parcellation
As a first step, let’s define the regions we want to extract the signal from. For this we can use nilearn’s great “dataset” function. How about the Harvard-Oxford Atlas? At first, we download the atlas and corresponding data via the fetch_atlas_harvard_oxford. Please note that we will use the default version but one can also specify the probabilities and resolution.

In [ ]:
from nilearn import datasets
atlas_ho = datasets.fetch_atlas_harvard_oxford('cort-maxprob-thr25-2mm')

In [ ]:
from nilearn import plotting
# Location of HarvardOxford parcellation atlas
atlas_file = atlas_ho.maps

# Visualize parcellation atlas
plotting.plot_roi(atlas_file, draw_cross=False, annotate=False, title="Harvard-Oxford Parcellation Atlas");

In [ ]:
# Load labels for each atlas region
labels = atlas_ho.labels[1:]
labels[:10]

## Extract timeseries to construct a functional connectome

In [ ]:
from nilearn.input_data import NiftiLabelsMasker
masker = NiftiLabelsMasker(labels_img=atlas_file, standardize=True, verbose=1)
time_series = masker.fit_transform(nifti_filename_new, confounds=confounds_filename)

## Compute and display the correlation matrix



In [ ]:
from nilearn.connectome import ConnectivityMeasure
correlation_measure = ConnectivityMeasure(kind='correlation')
correlation_matrix = correlation_measure.fit_transform([time_series])[0]

# Mask the main diagonal for visualization:
np.fill_diagonal(correlation_matrix, 0)

# Plot correlation matrix - note: matrix is ordered for block-like representation
plotting.plot_matrix(correlation_matrix, figure=(10, 8), labels=labels,
                     vmax=0.8, vmin=-0.8, reorder=True);

## Probabilistic atlas¶


In [ ]:
# Path to MSDL atlas
msdl_atlas = datasets.fetch_atlas_msdl()

# Extract only default mode network nodes
dmn_nodes = image.index_img(msdl_atlas.maps, [3, 4, 5, 6])
language_nodes = image.index_img(msdl_atlas.maps, [27,28,29,30,31])


# Plot MSDL probability atlas
plotting.plot_prob_atlas(language_nodes, cut_coords=(0, -60, 29), draw_cross=False,
                         annotate=False, title="Language nodes in MSDL atlas")

In [ ]:
from nilearn.input_data import NiftiMapsMasker
masker = NiftiMapsMasker(maps_img=msdl_atlas.maps, standardize=True, verbose=1,
                         memory="nilearn_cache", memory_level=2)

In [ ]:
# Extract the signal from the regions
time_series = masker.fit_transform(nifti_filename_new, confounds=confounds_filename)

# Compute the correlation matrix
correlation_matrix= correlation_measure.fit_transform([time_series])[0]

# Mask the main diagonal for visualization
np.fill_diagonal(correlation_matrix, 0)

In [ ]:
# CSV containing label and coordinate of MSDL atlas
msdl_labels = msdl_atlas.labels
msdl_coords = msdl_atlas.region_coords

# Plot the correlation matrix
plotting.plot_matrix(correlation_matrix, figure=(10, 8), labels=msdl_labels,
                     vmax=0.8, vmin=-0.8, reorder=True)

In [ ]:
plotting.plot_connectome(correlation_matrix, msdl_coords, edge_threshold="95%",
                         colorbar=True)

In [ ]:
plotting.view_connectome(correlation_matrix, msdl_coords, edge_threshold="95%", edge_cmap='bwr',
                         symmetric_cmap=True, linewidth=6.0, node_size=3.0)

## Seed-to-ROI mapping

In [ ]:
for label, coord in zip(msdl_atlas.labels[27:31], msdl_atlas.region_coords[27:31]):
    print(f"{label}: {coord}")

In [ ]:
from nilearn.input_data import NiftiSpheresMasker

# Sphere radius in mm
sphere_radius = 8

# Sphere center in MNI-coordinate
sphere_coords = [(-48, 25, 5)]

seed_masker = NiftiSpheresMasker(sphere_coords, radius=sphere_radius, detrend=True,
                                 standardize=True, low_pass=0.1, high_pass=0.01,
                                 t_r=2.0, verbose=1, memory="nilearn_cache", memory_level=2)

In [ ]:
# Extract the signal from the regions
seed_time_series = seed_masker.fit_transform(nifti_filename_new, confounds=confounds_filename)
from nilearn.input_data import NiftiMasker

brain_masker = NiftiMasker(smoothing_fwhm=6, detrend=True, standardize=True,
                           low_pass=0.1, high_pass=0.01, t_r=2., verbose=1,
                           memory="nilearn_cache", memory_level=2)

brain_time_series = brain_masker.fit_transform(nifti_filename_new, confounds=confounds_filename)


In [ ]:
seed_based_correlations = np.dot(brain_time_series.T, seed_time_series)
seed_based_correlations /= seed_time_series.shape[0]

## Plotting the seed-based correlation map¶


In [ ]:
seed_based_correlation_img = brain_masker.inverse_transform(seed_based_correlations.T)

conn_filename = Path(
    output_dir) / f"Seed-based_correlations_Broca.nii.gz"

seed_based_correlation_img.to_filename(conn_filename)

display = plotting.plot_stat_map(seed_based_correlation_img, threshold=0.333,
                                 cut_coords=sphere_coords[0])
display.add_markers(marker_coords=sphere_coords, marker_color='black',
                    marker_size=200)

In [ ]:
# print contrast
from ipywidgets import interact, interactive, fixed, interact_manual
from IPython.display import display
import ipywidgets as widgets
from ipyniivue import NiiVue, ShowRender, SliceType

# Create NiiVue instance with specific settings
nv = NiiVue(
    loading_text="waiting",
    back_color=(1, 1, 1, 1),
    show_3d_crosshair=True,
    is_colorbar=True,
    multiplanar_show_render=ShowRender.ALWAYS,
)

# Set initial configuration
nv.set_radiological_convention(False)
nv.set_slice_type(SliceType.MULTIPLANAR)
nv.set_slice_mm(False)
nv.set_interpolation(True)

# Load 4D volume with paired HEAD and BRIK files
nv.load_volumes(
    [
        {
            "path": "./ILAE_NeuroimagingSchool/dataset/anat/sub-ILAEDemo001_ses-01_run-01_space-MNI_desc-preproc_T1w.nii.gz",
        },
        {
            "path": "./ILAE_NeuroimagingSchool/outputs/Seed-based_correlations_Broca.nii.gz",
            "colormap": "warm",
            "colormap_negative": "winter",
            "cal_min": 3,
            "cal_max": 6,
            "cal_min_neg": -6,
            "cal_max_neg": -3,
        },
    ]
)


# Hide colorbar for anatomical scan
nv.volumes[0].colorbar_visible = False

# Set initial overlay outline
nv.overlay_outline_width = 0.25

# High DPI checkbox
dpi_checkbox = widgets.Checkbox(
    value=True,
    description="High DPI",
    tooltip="Higher resolution for 'retina' displays",
)

# Negative colors checkbox
negative_checkbox = widgets.Checkbox(value=True, description="Negative Colors")

# Smooth checkbox
smooth_checkbox = widgets.Checkbox(
    value=False,
    description="Smooth",
    tooltip=(
        "Trilinear interpolation blurs data, "
        "but can change which voxels survive a threshold"
    ),
)

# World space checkbox
world_checkbox = widgets.Checkbox(value=False, description="World Space")


# Outline width slider
outline_slider = widgets.IntSlider(
    min=0, max=4, value=1, description="Outline", continuous_update=True, readout=False
)

# Alpha mode dropdown
alpha_dropdown = widgets.Dropdown(
    options=[
        ("Restrict colorbar to range", 0),
        ("Colorbar from 0, transparent subthreshold", 1),
        ("Colorbar from 0, translucent subthreshold", 2),
    ],
    value=0,
    description="Alpha Mode:",
)

# threshold sliders

pos_stat_range = widgets.IntRangeSlider(
    value=[0,1],
    min=1,
    max=10,
    step=1,
    description='+Threshold:',
    continuous_update=True,
    readout=False,
)

neg_stat_range = widgets.IntRangeSlider(
    value=[0, 1],
    min=1,
    max=10,
    step=1,
    description='-Threshold:',
    continuous_update=True,
    readout=False,
)
# Location display
location_output = widgets.HTML(value="&nbsp;")

## Setup Event Handlers

def on_dpi_change(change):
    """Handle DPI checkbox changes."""
    nv.set_high_resolution_capable(change["new"])

def on_negative_change(change):
    """Handle negative colormap checkbox changes."""
    neg_stat_range.disabled = not change["new"]
    if change["new"]:
        nv.set_colormap_negative(nv.volumes[1].id, "winter")
    else:
        nv.set_colormap_negative(nv.volumes[1].id, "")

def on_smooth_change(change):
    """Handle smooth interpolation checkbox changes."""
    nv.set_interpolation(not change["new"])


def on_world_change(change):
    """Handle world space checkbox changes."""
    nv.set_slice_mm(change["new"])


def on_outline_change(change):
    """Handle outline width slider changes."""
    nv.overlay_outline_width = 0.25 * change["new"]


def on_alpha_mode_change(change):
    """Handle alpha mode dropdown changes."""
    nv.volumes[1].colormap_type = change["new"]

def on_pos_stat_change(change):
    """Set threshold for statistical overlay."""
    low, high = change["new"]
    nv.volumes[1].cal_min = low
    nv.volumes[1].cal_max = high 

def on_neg_stat_change(change):
    """Set threshold for statistical overlay."""
    low, high = change["new"]
    nv.volumes[1].cal_min_neg = -low
    nv.volumes[1].cal_max_neg = -high 

@nv.on_location_change
def handle_location_change(data):
    """Update location display when crosshair moves."""
    location_output.value = f"&nbsp;&nbsp;{data['string']}"

# Attach event handlers
dpi_checkbox.observe(on_dpi_change, names="value")
negative_checkbox.observe(on_negative_change, names="value")
smooth_checkbox.observe(on_smooth_change, names="value")
world_checkbox.observe(on_world_change, names="value")
outline_slider.observe(on_outline_change, names="value")
alpha_dropdown.observe(on_alpha_mode_change, names="value")

# Initialize values
on_outline_change({"new": outline_slider.value})
on_alpha_mode_change({"new": alpha_dropdown.value})

pos_stat_range.observe(on_pos_stat_change, names="value")
neg_stat_range.observe(on_neg_stat_change, names="value")
## Display All

# Organize controls
controls_row1 = widgets.HBox(
    [dpi_checkbox, negative_checkbox, smooth_checkbox, world_checkbox]
)
controls_row2 = widgets.HBox([neg_stat_range, pos_stat_range])
controls_row3 = widgets.HBox([outline_slider, alpha_dropdown])

# Create main layout
controls = widgets.VBox(
    [controls_row1, controls_row2, controls_row3, location_output]
)

# Display everything
display(widgets.VBox([controls, nv]))
